In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Executor クイックスタート

<Accordion>
<AccordionItem title="パッケージ・バージョン">

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以降の使用をお勧めします。

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

[Sampler](/guides/get-started-with-sampler) プリミティブと同様に、Executor は量子回路の実行から出力レジスターをサンプリングしますが、組み込みのエラー抑制や緩和機能はありません。代わりに、[指示型実行モデル](/guides/directed-execution-model)の一部として、設計の意図をクライアント側でキャプチャし、コストのかかる回路バリアントの生成をサーバー側にシフトします。Executor は、回路アノテーションとオプションで指定されたディレクティブに従い、パラメーター値を生成してバインドし、バインドされた回路をハードウェア上で実行し、実行結果とメタデータを返します。暗黙的な判断は一切行わず、完全な制御と透明性を提供します。

> **Note:** Qiskit パッケージには、Executor プリミティブの基底クラスがまだありません。

## 始める前に
このページのコード例の一部では、Samplomatic パッケージの一部である `samplex` を使用しています。そのため、これらのコード・ブロックを実行する前に、以下のコード・ブロックに示すように Samplomatic をインストールする必要があります。詳細については、[Samplomatic ドキュメント](https://qiskit.github.io/samplomatic)を参照してください。

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build

# Initialize the service and choose a backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

In [2]:
print(backend)

<IBMBackend('ibm_fez')>


### 2. Create and transpile a circuit

You need at least one circuit to use the Executor primitive.  It can optionally have parameters.

In [3]:
# Generate the circuit
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.h(1)

# Using `measure_all` automatically creates the necessary
# classical registers.
circuit.measure_all()

The circuit needs to be transformed to only use instructions supported by the QPU (referred to as *instruction set architecture (ISA)* circuits). Use the transpiler to do this.

In [4]:
# Transpile the circuit
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)
isa_circuit = preset_pass_manager.run(circuit)

### 2. 回路の作成とトランスパイル
Executor プリミティブを使用するには、少なくとも 1 つの回路が必要です。オプションでパラメーターを持つことができます。

In [5]:
# Initialize an empty program
program = QuantumProgram(shots=25)

# Append the circuit to the program
program.append_circuit_item(isa_circuit)

回路は、QPU でサポートされている命令のみを使用するように変換する必要があります（*命令セット・アーキテクチャー（ISA）回路*と呼ばれます）。これにはトランスパイラーを使用します。

In [6]:
# Generate a boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
boxes_pm = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)

boxed_circuit = boxes_pm.run(isa_circuit)
boxed_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/get-started-with-executor/extracted-outputs/245a4574-3ce9-4f77-98c8-af32cde8ac01-0.svg" alt="Output of the previous code cell" />

### 3. `QuantumProgram` の初期化
ワークロードを使用して `QuantumProgram` を初期化します。`QuantumProgram` は `QuantumProgramItems` で構成されています。通常、各アイテムは回路、パラメーター値のセット、およびオプションで回路の内容をランダム化する `samplex` で構成されます。詳細については、[Executor の入力と出力](/guides/executor-input-output)を参照してください。

次のセルは `QuantumProgram` を初期化し、25 ショットを実行するよう指定します。次に、トランスパイルされたターゲット回路を追加します。

In [7]:
# Build the template circuit and the samplex
template_circuit, samplex = build(boxed_circuit)

# Append the template circuit and samplex as a `samplex_item`
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    shape=(num_randomizations := 20,),
)

### 4. オプション：ゲートと測定をアノテーション付きボックスにグループ化する
命令をボックスにグループ化してアノテーションを付けることが、意図を指定する主要な方法です。次の例では、`generate_boxing_pass_manager` とそのツワーリング・パラメーターを使用して、2-Qubit ゲートと測定をボックスにグループ化し、ツワーリング・アノテーションを適用します。

In [8]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)
job

<RuntimeJobV2('d8286580bvlc73d1vmsg', 'executor')>

In [9]:
# Retrieve the result
result = job.result()

### 6. Executor の呼び出しと結果の取得
デフォルト・オプションで `Executor` プリミティブを使用して、IBM&reg; バックエンドで `QuantumProgram` を実行します。利用可能なオプションについては、[Executor オプション](/guides/executor-options)を参照してください。